In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")
os.environ["LANGSMITH_TRACING"] = "true"

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

1. Environment Setup

In [ ]:
pip install sentence-transformers langchain openai hdbscan

2. Load and Embed the Responses

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
responses = [
    "The checkout process was confusing and long.",
    "Customer support was very helpful.",
    "I loved the fast delivery.",
    "The website UI needs improvement.",
    "Delivery was quick and seamless.",
    "Support team resolved my issue in minutes."
]
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(responses)


3. Clustering the Responses

In [ ]:
import hdbscan
from collections import defaultdict

In [ ]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
clusters = clusterer.fit_predict(embeddings)
clustered_responses = defaultdict(list)
for idx, cluster_id in enumerate(clusters):
    if cluster_id != -1:
        clustered_responses[cluster_id].append(responses[idx])

Now, each cluster contains semantically similar responses.

4. LangChain Map-Reduce Summarization

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.chains.summarize import load_summarize_chain
from langchain.docstore.document import Document

In [ ]:
llm = ChatOpenAI(temperature=0, model_name="gpt-4")
cluster_summaries = []
for cluster_id, texts in clustered_responses.items():
    documents = [Document(page_content=text) for text in texts]
    chain = load_summarize_chain(llm, chain_type="map_reduce")
    summary = chain.run(documents)
    cluster_summaries.append(summary)

5. Reduce Step: Final Global Summary

In [ ]:
final_docs = [Document(page_content=s) for s in cluster_summaries]
final_summary_chain = load_summarize_chain(llm, chain_type="stuff")
final_summary = final_summary_chain.run(final_docs)

In [ ]:
print("Final Summary:\n", final_summary)

Sample Output

In [ ]:
Final Summary:
Customers appreciated fast delivery and helpful customer support. However, some noted issues with the website interface and the checkout experience.

### Theme Labeling (Optional)
You can label each cluster with a descriptive theme using an LLM:

In [ ]:
theme_prompt = """
Given the following list of user responses:
{responses}
Identify a short, clear theme or topic that represents all of them.
"""

In [ ]:
for cluster_id, texts in clustered_responses.items():
    joined_text = "\n".join(texts)
    theme = llm.predict(theme_prompt.format(responses=joined_text))
    print(f"Cluster {cluster_id} Theme: {theme}")

#### Benefits of This Approach
- Thematic Summarization: Captures context-specific insights.
- Scalability: Efficiently handles large volumes of text.
- Interpretability: Provides clarity on what each group of responses represents.
- Modular: Swap models or clustering methods without altering the core logic.
#### Challenges and Considerations
- Cluster Quality: Requires tuning (e.g., min_cluster_size) to avoid over/under clustering.
- LLM Costs: Summarizing clusters can be expensive with large volumes.
- Theme Naming Accuracy: May need human verification.
#### Conclusion
This clustering-based summarization pipeline leverages the power of semantic embeddings and LangChain’s map-reduce framework to produce high-quality, interpretable summaries. It’s particularly well-suited for analyzing open-ended text data in customer research, social listening, and employee surveys.

With a few modifications, this pipeline can also be extended to multilingual datasets, incorporate sentiment analysis, or support real-time feedback summarization.